# Certified polynomial one-jet reduction benchmarks

This notebook compares interval Jacobians, unreduced polynomial Jacobians, and three sound support-reduction policies. The 100-dimensional target is the saved Poisson PINN checkpoint; running the benchmark never retrains it. Reduced terms are not discarded: their componentwise remainder is propagated rigorously and attached as fresh pointwise residual symbols at the output. Tightness is assessed by the absolute and relative widths of both the final certified $L^2$ and $W^{1,2}$ norm intervals. Before integration, we also report the mean, maximum, and relative mean componentwise widths of the full Jacobian enclosure.

In [1]:
from collections import Counter
from math import sqrt
from pathlib import Path
from time import perf_counter
import torch
from intervalnets import (IntervalTensor, PZIntegrationCell, enable_interval_eval,
    integrate_pz_onejet_squared, integrate_pz_value_squared,
    load_tanh_mlp_checkpoint)
torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)
enable_interval_eval()
repo_root = Path.cwd()
while not (repo_root / 'notebooks' / 'checkpoints').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
CHECKPOINT = repo_root / 'notebooks' / 'checkpoints' / 'pinn_100d_poisson.pt'

In [2]:
def make_model(input_dim, hidden=(50, 50, 50), seed=20260731):
    torch.manual_seed(seed)
    layers, previous = [], input_dim
    for width in hidden:
        layers += [torch.nn.Linear(previous, width), torch.nn.Tanh()]
        previous = width
    layers.append(torch.nn.Linear(previous, 1))
    return torch.nn.Sequential(*layers)

def norm_interval(squared):
    return (sqrt(max(0.0, float(squared.lower))), sqrt(max(0.0, float(squared.upper))))

def interval_metrics(bounds, prefix):
    lower, upper = map(float, bounds)
    absolute_width = upper - lower
    relative_width = absolute_width / upper if upper > 0.0 else 0.0
    return {
        f'{prefix}_lower': lower,
        f'{prefix}_upper': upper,
        f'{prefix}_absolute_width': absolute_width,
        f'{prefix}_relative_width': relative_width,
    }

def jacobian_width_metrics(enclosure):
    lower = torch.as_tensor(enclosure.lower)
    upper = torch.as_tensor(enclosure.upper)
    widths = upper - lower
    scales = torch.maximum(lower.abs(), upper.abs())
    relative_widths = torch.where(scales > 0.0, widths / scales, 0.0)
    return {
        'J_mean_component_width_before_integration': float(widths.mean()),
        'J_max_component_width_before_integration': float(widths.max()),
        'J_relative_mean_component_width_before_integration': float(relative_widths.mean()),
    }

def benchmark(model, box, strategy='topk', **kwargs):
    cell = PZIntegrationCell.from_affine_box(box)
    start = perf_counter()
    traced = model.eval_pz_onejet(cell.domain, return_trace=True,
        reduction_strategy=strategy, **kwargs)
    forward_s = perf_counter() - start
    enclosure = traced.final.J.interval_enclosure()
    jacobian_metrics = jacobian_width_metrics(enclosure)
    start = perf_counter()
    l2_squared = integrate_pz_value_squared(traced.final.Y, cell)
    l2_integration_s = perf_counter() - start
    start = perf_counter()
    w12_squared = integrate_pz_onejet_squared(traced.final, cell)
    w12_integration_s = perf_counter() - start
    return {
        'strategy': strategy, **kwargs, 'forward_s': forward_s,
        'L2_integration_s': l2_integration_s,
        'W12_integration_s': w12_integration_s,
        'total_s': forward_s + w12_integration_s,
        'J_terms': len(traced.final.J.terms),
        'J_degree': max(map(sum, traced.final.J.terms), default=0),
        'noise': traced.final.J.num_noise,
        **jacobian_metrics,
        **interval_metrics(norm_interval(l2_squared), 'L2'),
        **interval_metrics(norm_interval(w12_squared), 'W12'),
        'trace': traced.records,
    }

## Small-network exact reference
The unreduced path is practical here and provides the polynomial reference endpoint.

In [3]:
small_model = make_model(8, hidden=(10, 10))
small_box = IntervalTensor.from_bounds([-0.15] * 8, [0.15] * 8)
small_exact = benchmark(small_model, small_box, strategy='none', reduce=False)
small_reduced = [benchmark(small_model, small_box, strategy=s, max_terms=24,
    max_degree=2, pca_rank=3, pca_candidates=24) for s in ('topk', 'degree', 'pca')]
[{k: v for k, v in row.items() if k != 'trace'} for row in [small_exact, *small_reduced]]

[{'strategy': 'none', 'reduce': False, 'forward_s': 0.012567275999572303, 'L2_integration_s': 0.001020986999719753, 'W12_integration_s': 0.017564858999321586, 'total_s': 0.03013213499889389, 'J_terms': 142, 'J_degree': 2, 'noise': 36, 'J_mean_component_width_before_integration': 0.02382887976904387, 'J_max_component_width_before_integration': 0.033046349912150164, 'J_relative_mean_component_width_before_integration': 0.3906042968692145, 'L2_lower': 0.0021888955870275227, 'L2_upper': 0.0022656904960123648, 'L2_absolute_width': 7.679490898484208e-05, 'L2_relative_width': 0.03389470411779623, 'W12_lower': 0.0025362124712323794, 'W12_upper': 0.0027754193636663474, 'W12_absolute_width': 0.00023920689243396792, 'W12_relative_width': 0.08618765710345626}, {'strategy': 'topk', 'max_terms': 24, 'max_degree': 2, 'pca_rank': 3, 'pca_candidates': 24, 'forward_s': 0.009422064999853319, 'L2_integration_s': 0.0007209909999801312, 'W12_integration_s': 0.0035069789992121514, 'total_s': 0.01292904399906

## Saved 100D Poisson PINN: 100–50–50–50–1
The trained candidate is loaded from the committed checkpoint; there is no optimizer or training loop in this notebook. All policies below therefore certify exactly the same saved PINN weights. They preserve a Jacobian polynomial core. The interval result is the speed/looseness baseline. The primary comparison quantities are the absolute and relative widths of both norm intervals; the Jacobian width diagnostics measure tightness before the integration step.

In [4]:
model = load_tanh_mlp_checkpoint(CHECKPOINT)
assert sum(parameter.numel() for parameter in model.parameters()) == 10201
print({'loaded_checkpoint': str(CHECKPOINT.relative_to(repo_root)), 'training_steps': 0})
box = IntervalTensor.from_bounds([-0.1] * 100, [0.1] * 100)
configs = [
    ('topk-32', 'topk', dict(max_terms=32)),
    ('topk-64', 'topk', dict(max_terms=64)),
    ('topk-96', 'topk', dict(max_terms=96)),
    ('degree-64', 'degree', dict(max_terms=64, max_degree=2)),
    ('pca-64', 'pca', dict(max_terms=64, pca_rank=4, pca_candidates=32)),
]
rows = []
for label, strategy, kwargs in configs:
    row = benchmark(model, box, strategy=strategy, **kwargs)
    row['label'] = label
    rows.append(row)
start = perf_counter()
interval_bound = model.sobolev_norm(box, p=2.0, order=1, method='interval')
interval_s = perf_counter() - start
interval_l2_bound = model.lpnorm(box, p=2.0, method='interval')
interval_jacobian = model.eval_jacobian(box)
interval_row = {
    'label': 'interval',
    'total_s': interval_s,
    **jacobian_width_metrics(interval_jacobian),
    **interval_metrics((interval_l2_bound.lower, interval_l2_bound.upper), 'L2'),
    **interval_metrics((interval_bound.lower, interval_bound.upper), 'W12'),
}
summary = [{k: v for k, v in row.items() if k != 'trace'} for row in rows]
[interval_row, *summary]

{'loaded_checkpoint': 'notebooks/checkpoints/pinn_100d_poisson.pt', 'training_steps': 0}


[{'label': 'interval', 'total_s': 0.21159809799974028, 'J_mean_component_width_before_integration': 17.548814954264703, 'J_max_component_width_before_integration': 20.8648129804914, 'J_relative_mean_component_width_before_integration': 1.9897599352068311, 'L2_lower': 0.0, 'L2_upper': 8.984143949899863e-35, 'L2_absolute_width': 8.984143949899863e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 1.0003260914983017e-33, 'W12_absolute_width': 1.0003260914983017e-33, 'W12_relative_width': 1.0}, {'strategy': 'topk', 'max_terms': 32, 'forward_s': 0.5200860679997277, 'L2_integration_s': 0.01422789299977012, 'W12_integration_s': 0.6923924359998637, 'total_s': 1.2124785039995913, 'J_terms': 132, 'J_degree': 1, 'noise': 350, 'J_mean_component_width_before_integration': 17.112409763051467, 'J_max_component_width_before_integration': 20.290219948868312, 'J_relative_mean_component_width_before_integration': 1.981436965912812, 'L2_lower': 0.0, 'L2_upper': 3.0138433436891297e-35, 'L2_absol

In [5]:
assert all(row['total_s'] < 3.0 for row in rows), summary
start = perf_counter()
public_l2_bound = model.pz_l2norm(box)
public_l2_s = perf_counter() - start
start = perf_counter()
public_w12_bound = model.pz_sobolev_norm(box, order=1)
public_w12_s = perf_counter() - start
{
    'public_default_L2_s': public_l2_s,
    'public_default_W12_s': public_w12_s,
    **interval_metrics((public_l2_bound.lower, public_l2_bound.upper), 'public_default_L2'),
    **interval_metrics((public_w12_bound.lower, public_w12_bound.upper), 'public_default_W12'),
}

{'public_default_L2_s': 0.0655189649996828, 'public_default_W12_s': 2.09917937299997, 'public_default_L2_lower': -5e-324, 'public_default_L2_upper': 3.013843343689131e-35, 'public_default_L2_absolute_width': 3.013843343689131e-35, 'public_default_L2_relative_width': 1.0, 'public_default_W12_lower': -5e-324, 'public_default_W12_upper': 9.61182469677171e-34, 'public_default_W12_absolute_width': 9.61182469677171e-34, 'public_default_W12_relative_width': 1.0}

## Layer diagnostics
The activation rows expose where support generation and certified tail growth occur. The benchmark-level Jacobian widths above are computed after the complete one-jet has been constructed but before either squared integral is evaluated. They use the full enclosure $J_{\mathrm{core}}+[-R,R]$, not merely the reduction remainder. For each entry, the relative width is $(\overline J_{ij}-\underline J_{ij})/\max(|\underline J_{ij}|,|\overline J_{ij}|)$, with exact-zero entries assigned zero; the reported relative mean is the mean of these componentwise ratios.

In [6]:
chosen = next(row for row in rows if row['label'] == 'topk-96')
[{
  'layer': record.layer_type, 'seconds': record.elapsed_s,
  'Y_terms': record.summary['Y']['term_count'],
  'J_terms': record.summary['J']['term_count'],
  'J_degree': record.summary['J']['max_degree'],
  'remainder_mean_radius': record.summary['J']['remainder_mean_radius'],
} for record in chosen['trace']]

[{'layer': 'Input', 'seconds': 0.0, 'Y_terms': 100, 'J_terms': 0, 'J_degree': 0, 'remainder_mean_radius': 0.0}, {'layer': 'Linear', 'seconds': 0.0036662960001194733, 'Y_terms': 100, 'J_terms': 0, 'J_degree': 0, 'remainder_mean_radius': 0.0}, {'layer': 'Tanh', 'seconds': 0.026920215000245662, 'Y_terms': 150, 'J_terms': 96, 'J_degree': 1, 'remainder_mean_radius': 0.026446936553910286}, {'layer': 'Linear', 'seconds': 0.009323897999820474, 'Y_terms': 150, 'J_terms': 96, 'J_degree': 1, 'remainder_mean_radius': 0.15294793951109945}, {'layer': 'Tanh', 'seconds': 0.5793782150003608, 'Y_terms': 200, 'J_terms': 86, 'J_degree': 1, 'remainder_mean_radius': 0.17384077899371955}, {'layer': 'Linear', 'seconds': 0.011454362999756995, 'Y_terms': 200, 'J_terms': 86, 'J_degree': 1, 'remainder_mean_radius': 1.0390345844288842}, {'layer': 'Tanh', 'seconds': 0.6544056239999918, 'Y_terms': 250, 'J_terms': 96, 'J_degree': 1, 'remainder_mean_radius': 1.0679993370671088}, {'layer': 'Linear', 'seconds': 0.012058

## Interpretation

- Top-k is the cheapest reduction and gives a direct runtime/tightness knob.
- For both $L^2$ and $W^{1,2}$, the primary final tightness diagnostic is $U-L$ for the certified norm interval $[L,U]$. The reported relative width is $(U-L)/U$ (and is defined as zero when $U=0$).
- The mean, maximum, and relative mean Jacobian component widths are complementary pre-integration diagnostics: they show how much tightness has already been lost in the image enclosure, before squaring and integration can add further overestimation. The relative mean averages the entrywise width divided by the largest endpoint magnitude, so it is scale-normalized and lies between zero and two.
- In the target experiment every method currently has $L=0$ for both norms, hence every relative norm width is $100\%$. Here a smaller upper endpoint happens to equal a smaller absolute width, but it does not constitute an improvement in relative precision.
- The target-network relative mean Jacobian widths are close to two. This says that most component intervals straddle zero and are nearly symmetric relative to their endpoint magnitude. The absolute mean and maximum widths therefore remain the more discriminating Jacobian diagnostics in this experiment.
- Degree capping matters once higher-degree terms survive the importance ranking; on narrow boxes it can coincide with top-k.
- PCA is certified because the projected generators are intervalized in PCA coordinates and the orthogonal residual is bounded componentwise. Its SVD and added pointwise generators must earn their cost empirically.
- The unreduced polynomial path is intentionally limited to smaller networks: it diagnoses genuine monomial growth rather than hiding it behind interval propagation.